# Notebook 10 — Transfer: statistics across training seeds

*Corrected version of the original `10-transfer-significance (1).ipynb`. The training now
happens in Notebook_09; this notebook only reads its result files.*

**What this notebook does.** It reads the per-seed transfer results and reports, for each
direction, K and view: the mean, standard deviation and 95% confidence interval of every
method across training seeds, paired differences with two-sided tests, and equivalence
tests. It then lines the new numbers up with the old ones.

**Input.** `transfer_seed<seed>.json` from Notebook_09 (in the output folder, or attached on
Kaggle). **Output.** `transfer_significance_summary.json`. No GPU needed.

### What was corrected
1. **Two-sided tests.** The original reported one-sided p-values ("MAML > Static"), e.g.
   p = 0.53, without saying so. All tests here are two-sided.
2. **Equivalence is tested, not assumed.** "MAML is about equal to Static" is supported only
   if a TOST equivalence test for +/- 0.02 ROC-AUC gives p < 0.05.
3. **All K and both views.** The original tested K = 50 only, and only with the leaky
   target projection.
4. **Seed-to-seed variability** (SD across training seeds) is reported for every method.
5. **Scratch is compared properly** (a model trained on the support windows), and the gap to
   the Target-static upper reference is reported.
6. PCA explained variance: full-data fit against support-only fits.

In [ ]:
import os, sys

def _find_common():
    """Find maml_common.py: this folder when run locally, /kaggle/input on Kaggle."""
    for root in [os.getcwd(), "/kaggle/input"]:
        if os.path.isdir(root):
            for d, _, files in os.walk(root):
                if "maml_common.py" in files:
                    return d
    raise FileNotFoundError("maml_common.py not found. Run from the Corrected-Notebooks folder, "
                            "or attach that folder to Kaggle as a Dataset.")

sys.path.insert(0, _find_common())
import maml_common as sc
SMOKE = os.environ.get("SMAP_SMOKE") == "1"     # tiny settings for testing only
OUT = sc.output_dir(smoke=SMOKE)
print("shared code:", sc.__file__)
print("outputs go to:", OUT)
print("code version:", sc.git_commit())

import json, glob, numpy as np

## 1 — Load the per-seed results

In [ ]:
def find_results():
    pats = [os.path.join(OUT, "transfer_seed*.json")] + (["/kaggle/input/**/transfer_seed*.json"] if os.path.isdir("/kaggle/input") else [])
    files = {}
    for pat in pats:
        for f in glob.glob(pat, recursive=True):
            if SMOKE == f.endswith("_SMOKE.json"):
                files.setdefault(os.path.basename(f), f)
    return sorted(files.values())

files = find_results()
runs = {json.load(open(f))["train_seed"]: json.load(open(f)) for f in files}
assert runs, "no transfer_seed*.json found: run Notebook_09 first"
CFG = next(iter(runs.values()))["config"]
print("training seeds:", sorted(runs), "| directions:", CFG["directions"], "| K:", CFG["k_shots"])

## 2 — Per-seed values and summaries

In [ ]:
METHODS = ["MAML-transfer", "MAML-transfer (0 steps)", "Static-transfer", "Static-transfer (0 steps)",
           "Scratch", "Scratch (legacy)", "LSTM-AE untrained (floor)", "Target-static (upper reference)"]

def per_seed(run, direction, k, view, method, metric="roc_auc"):
    vals = [run["results"][f"{direction}|{k}|{s}"][view].get(method, {}).get(metric) for s in CFG["support_seeds"]]
    vals = [v for v in vals if v is not None]
    return float(np.mean(vals)) if vals else None

out = {"methods": {}, "paired": {}, "pca": {}}
fmt = lambda e: "n/a" if e["mean"] is None else f"{e['mean']:.4f}" + (f" (SD {e['sd']:.4f}) [{e['ci95'][0]:.3f}, {e['ci95'][1]:.3f}]" if "sd" in e else "")
for src, tgt in CFG["directions"]:
    direction = f"{src}->{tgt}"
    for view in ["leak_free", "legacy_full_target_fit"]:
        for k in CFG["k_shots"]:
            print(f"\n=== {direction}, K = {k}, view: {view} — ROC-AUC across {len(runs)} training seeds ===")
            vals = {m: [per_seed(r, direction, k, view, m) for r in runs.values()] for m in METHODS}
            for m in METHODS:
                e = sc.describe_values(vals[m])
                e["f1_at_tau"] = sc.describe_values([per_seed(r, direction, k, view, m, "f1_at_tau") for r in runs.values()])
                out["methods"][f"{direction}|{view}|{k}|{m}"] = e
                if e["mean"] is not None:
                    print(f"  {m:34s} {fmt(e)}")
            for a, b in [("MAML-transfer", "Static-transfer"), ("MAML-transfer", "Scratch"), ("Static-transfer", "Scratch"),
                         ("MAML-transfer", "MAML-transfer (0 steps)"), ("Static-transfer", "Static-transfer (0 steps)"),
                         ("Target-static (upper reference)", "MAML-transfer")]:
                if all(v is None for v in vals[a]):
                    continue
                pc = sc.paired_comparison(vals[a], vals[b], 0.02)
                out["paired"][f"{direction}|{view}|{k}|{a} minus {b}"] = pc
                if "ci95" in pc:
                    print(f"  {a} minus {b}: {pc['mean']:+.4f} [{pc['ci95'][0]:+.4f}, {pc['ci95'][1]:+.4f}], "
                          f"p(t, two-sided) {pc['t_test_p_two_sided']:.3f}, TOST p {pc.get('tost_p', float('nan')):.3f}")
    for k in CFG["k_shots"]:
        evk = [run["results"][f"{direction}|{k}|{s}"]["target_projection_explained_variance"] for run in runs.values() for s in CFG["support_seeds"]]
        out["pca"][f"{tgt}|support_only|K={k}"] = sc.describe_values(evk)
for p, v in next(iter(runs.values()))["full_fit_explained_variance"].items():
    out["pca"][f"{p}|full_normal_fit"] = v
print("\nPCA variance kept by 32 components:")
for k, v in out["pca"].items():
    print(f"  {k:28s} {v if isinstance(v, float) else fmt(v)}")

## 3 — Next to the old numbers

The old seeded run used K = 50, 6 seeds and the leaky projection (the "legacy" view here),
with one-sided tests. Its scratch numbers came from a single run of an essentially untrained
network ("Scratch (legacy)" here).

In [ ]:
OLD = {"wadi->swat": {"Target-static": 0.8116, "MAML": (0.7962, 0.0099), "Static": (0.7969, 0.0147),
                      "diff": (-0.0007, [-0.0252, 0.0239]), "scratch_single_run": {20: 0.7503, 50: 0.7563, 100: 0.6813}},
       "swat->wadi": {"Target-static": 0.7074, "MAML": (0.5742, 0.0378), "Static": (0.5953, 0.0051),
                      "diff": (-0.0211, [-0.0601, 0.0179]), "scratch_single_run": {20: 0.5039, 50: 0.4762, 100: 0.5052}},
       "pca_full_fit": {"swat": 0.9999, "wadi": 0.9733}}
for d in ["wadi->swat", "swat->wadi"]:
    if 50 not in CFG["k_shots"]:
        print("K = 50 not in this run; comparison skipped"); break
    o = OLD[d]
    for view in ["legacy_full_target_fit", "leak_free"]:
        m = out["methods"][f"{d}|{view}|50|MAML-transfer"]; s = out["methods"][f"{d}|{view}|50|Static-transfer"]
        pc = out["paired"].get(f"{d}|{view}|50|MAML-transfer minus Static-transfer", {})
        print(f"\n{d}, K = 50, {view}:")
        print(f"  MAML   old {o['MAML'][0]:.4f} (SD {o['MAML'][1]:.4f})  new {fmt(m)}")
        print(f"  Static old {o['Static'][0]:.4f} (SD {o['Static'][1]:.4f})  new {fmt(s)}")
        if "ci95" in pc:
            print(f"  MAML - Static old {o['diff'][0]:+.4f} {o['diff'][1]}  new {pc['mean']:+.4f} "
                  f"[{pc['ci95'][0]:+.4f}, {pc['ci95'][1]:+.4f}]")
    print(f"  Target-static old {o['Target-static']:.4f}  new {fmt(out['methods'][f'{d}|legacy_full_target_fit|50|Target-static (upper reference)'])}")
sc.save_json(os.path.join(OUT, f"transfer_significance_summary{'_SMOKE' if SMOKE else ''}.json"),
             {"smoke_test": SMOKE, "train_seeds": sorted(runs), "summary": out, "old_numbers": OLD})

## 4 — How to read the results

- The **leak-free** view is the valid few-shot test; the **legacy** view reproduces the
  original set-up only for comparison.
- A difference counts only if its 95% interval excludes 0. "About equal" needs TOST p < 0.05.
- A large SD across training seeds means a single run could have landed anywhere in that
  range.